# 05. Map transfer through π

π appears to carry areal position on the cortical hierarchy, along with the properties that vary
across that axis. Properties that vary through the cortical depth do not come across.

Nothing here requires re-fitting. Each result routes a mouse map through the frozen coupling and
correlates it with an independent human map, so the notebook recomputes them.

Two smooth brain maps correlate with each other readily, so the test is whether this coupling
carries the signal. Every test below
uses `translation_spin_null`, which spins the mouse map and routes it through the real π. That
preserves both spatial autocorrelation and the coupling, and breaks only the specific mouse→human
correspondence. Nulls that shuffle region labels destroy spatial autocorrelation and will report
relationships that do not survive a spin.

Before running: `python scripts/fetch_data.py`. Logs are provenance-checked before use.

In [ ]:
import importlib.util, json, subprocess, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from otter.data import load_pi, pi_provenance
from otter.eval.nulls import translation_spin_null

pi = load_pi()
PROV = pi_provenance()
LOGS = ROOT / 'outputs' / 'logs'
print(f"coupling {pi.shape[0]:,} x {pi.shape[1]:,}   {PROV['pi_file']}   sha {PROV['pi_sha256'][:16]}...")

PUBLISHED = {
    'myelin from mouse T1w:T2w (r)':      (0.47, 0.02),
    'myelin from mouse cytoarch (r)':     (0.47, 0.02),
    'routed Schaefer regions':            (388,  1),
    'networks top-matching homologue':    (6,    0),
    'network spin p':                     (0.002, 0.002),
    'marker mean r':                      (0.23, 0.02),
    'contrast mean r':                    (0.07, 0.02),
}

def check(name, value):
    exp, tol = PUBLISHED[name]
    ok = abs(value - exp) <= tol
    print(f"  [{'ok ' if ok else 'FAIL'}] {name:34s} computed {value:.4g}   reported {exp}")
    return ok


def verified_log(fname):
    '''Read a log only after confirming which coupling produced it.'''
    d = json.loads((LOGS / fname).read_text())
    sha = d.get('pi_sha256') if isinstance(d, dict) else None
    if sha is None:
        print(f"  {fname}: no coupling provenance recorded")
    elif sha != PROV['pi_sha256']:
        raise RuntimeError(f"{fname} was built on a different coupling ({sha[:16]}...)")
    else:
        print(f"  {fname}: provenance verified")
    return d


def load_experiment(relpath):
    '''Import an experiment module by path (their filenames start with digits).'''
    p = ROOT / 'experiments' / relpath
    spec = importlib.util.spec_from_file_location(p.stem.lstrip('0123456789_'), p)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

## 1. Microstructural translation

Two independent mouse measurements, a T1w:T2w myelin proxy and cytoarchitectural type, are routed
through π and compared against the human HCP myelin map, which the model never saw.

Translation is both connectional and microstructural. The values are recomputed here from the
raw maps.

In [ ]:
F = load_experiment('fulcher_2019_multimodal_gradient/01_gradient_validation.py')

parcel_acr = F.load_mouse_parcel_acronyms()
node_region = F.load_human_node_region()
myelin_reg = F.load_human_myelin()

def route_modality(value_by_acr):
    vec = np.array([value_by_acr.get(a, np.nan) for a in parcel_acr])
    mask = np.isfinite(vec)
    return vec, mask, F.aggregate_to_regions(F.route_through_pi(vec, pi, mask), node_region)

t1t2_vec, t1t2_mask, pred_t1t2 = route_modality(F.load_mouse_t1t2())
cyto_vec, cyto_mask, pred_cyto = route_modality(F.load_mouse_cytoarch())

r_t1t2 = F._corr(pred_t1t2, myelin_reg)
r_cyto = F._corr(pred_cyto, myelin_reg)
n_regions = int(np.isfinite(pred_t1t2).sum())

print(f"routed territory: {n_regions} of 400 Schaefer regions\n")
print(f"mouse T1w:T2w        -> human myelin   r = {r_t1t2[0]:+.3f}  (n = {r_t1t2[3]})")
print(f"mouse cytoarchitecture -> human myelin r = {r_cyto[0]:+.3f}  (n = {r_cyto[3]})\n")
check('myelin from mouse T1w:T2w (r)', abs(r_t1t2[0]))
check('myelin from mouse cytoarch (r)', abs(r_cyto[0]))
check('routed Schaefer regions', n_regions)

## 2. The transfer battery

The battery groups fourteen properties by their relation to the areal hierarchy: the macroscale
hierarchy maps themselves, properties that vary along that hierarchy, and properties orthogonal
to it (laminar contrasts, spatially uniform cell classes). The claim is a dissociation, so it
needs the negatives as much as the positives.

Each entry is a routed translation with a spin null, so the cost is modest. `RUN_EXPERIMENTS =
True` recomputes the battery from the raw data in tens of minutes; otherwise the logs are
provenance-checked.

In [ ]:
RUN_EXPERIMENTS = False

BATTERY = {
    'published_map_validation.json':      'validation/00_validate_published_maps.py',
    'biccn_contrast_reframe.json':        'biccn_2023_cell_types/03_contrast_reframe.py',
    'hodge_areal_type_reframe.json':      'hodge_2019_cortical_layers/03_areal_type_reframe.py',
    'margulies_2016_gradient.json':       None,      # written by the TransBrain benchmark
    'fulcher_2019_gradient.json':         None,      # written by the section-5 coverage nulls
}

if RUN_EXPERIMENTS:
    for log, rel in BATTERY.items():
        if rel is None:
            print(f'{log}: produced elsewhere, skipping'); continue
        print(f're-running {rel} ...')
        r = subprocess.run([sys.executable, str(ROOT / 'experiments' / rel)], cwd=str(ROOT),
                           capture_output=True, text=True)
        print((r.stdout or r.stderr)[-300:])

for log in BATTERY:
    verified_log(log)

In [ ]:
# Individual layer-marker genes (areal signal retained) versus layer contrasts built from the
# same genes (areal signal removed). The contrast is the control. If translation carried
# laminar rather than areal information, contrasts would survive.
# Granular L4 minus infragranular is the expected exception, because cortical granularity is
# the areal hierarchy, so that contrast does not remove the areal signal.
#
# Every correlation below is recomputed from data_external/mouse_genes.npy, human_genes.npy and
# the two gene lists, routed through the frozen coupling. The code path follows
# experiments/hodge_2019_cortical_layers/01_layer_marker_validation.py for the per-gene markers,
# 03_areal_type_reframe.py for the three layer contrasts, and 02_layer_marker_refined.py for the
# upper minus deep contrast. The three logs are still read, and the recomputed correlations are
# compared against them.
import pandas as pd

from otter.data import load_cached
from otter.data.atlas_regions import (ATLAS_PATHS, assign_atlas_labels,
                                      assign_atlas_labels_with_hemisphere)
from otter.eval.nulls import _route_normalized

mk = verified_log('hodge_2019_layer_markers.json')
areal = verified_log('hodge_areal_type_reframe.json')
refined = verified_log('hodge_2019_layer_markers_refined.json')

M, _ = load_cached('mouse', cache_dir=str(ROOT / 'outputs/anndata'))
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))
mouse_coords = M.var[['x', 'y', 'z']].to_numpy(float)

HODGE_MARKERS = ['Cux1', 'Cux2', 'Satb2', 'Rorb', 'Fezf2', 'Tbr1', 'Foxp2']
UPPER, GRANULAR, DEEP = ['Cux1', 'Cux2', 'Satb2'], ['Rorb'], ['Fezf2', 'Tbr1', 'Foxp2']

mouse_expr = np.load(ROOT / 'data_external/mouse_genes.npy')
mouse_genes = pd.read_csv(ROOT / 'data_external/mouse_gene_list.csv')
human_expr = np.load(ROOT / 'data_external/human_genes.npy')
human_genes = pd.read_csv(ROOT / 'data_external/human_gene_list.csv')
for E in (mouse_expr, human_expr):
    # a missing measurement takes its column mean
    colmean = np.nanmean(E, axis=0)
    nz = np.where(np.isnan(E))
    E[nz] = np.take(colmean, nz[1])

def m_col(sym):
    return int(mouse_genes[mouse_genes['gene_symbol'].str.lower() == sym.lower()].iloc[0].name)

def h_col(sym):
    return int(human_genes[human_genes['gene_symbol'].str.upper() == sym.upper()].iloc[0].name)

def zscore_safe(v):
    # the z-score of 01_layer_marker_validation.py and 02_layer_marker_refined.py
    sd = v.std()
    return np.zeros_like(v) if sd < 1e-9 else (v - v.mean()) / sd

def zscore_float(v):
    # the z-score of 03_areal_type_reframe.py, which casts to float64 first
    v = v.astype(float)
    s = np.nanstd(v)
    return (v - np.nanmean(v)) / (s if s > 1e-9 else 1.0)

def composite(expr, cols):
    return np.column_stack([zscore_safe(expr[:, c]) for c in cols]).mean(axis=1)

def areal_score(expr, cols):
    return np.column_stack([zscore_float(expr[:, c]) for c in cols]).mean(axis=1)

# Per-gene markers, over all 2094 human parcels. Mouse expression is z-scored, routed through
# pi, and correlated against observed human expression of the same gene. The null permutes the
# rows of pi, 200 trials per gene, from one generator seeded at 42 for the whole loop.
rng = np.random.default_rng(seed=42)
markers = []
for sym in HODGE_MARKERS:
    m_z = zscore_safe(mouse_expr[:, m_col(sym)])
    h_z = zscore_safe(human_expr[:, h_col(sym)])
    r_obs = float(pearsonr(m_z @ pi, h_z)[0])
    null = np.array([float(pearsonr(m_z @ pi[rng.permutation(pi.shape[0])], h_z)[0])
                     for _ in range(200)])
    markers.append((sym, r_obs, float((null >= r_obs).mean())))

# The cortical mask on the human side comes from Schaefer-400.
sch = assign_atlas_labels(H.var, 'schaefer_400', str(ROOT / ATLAS_PATHS['schaefer_400']))
sch = assign_atlas_labels_with_hemisphere(H.var, sch)
cortex = sch > 0

# The three layer contrasts, cortex only. Each mouse contrast is routed through pi as a
# transport-weighted average and correlated against the same contrast in human expression. The
# null spins the mouse map and routes it through the real pi, 1000 rotations, the count
# 03_areal_type_reframe.py uses.
contrasts = []
for label, pos, neg in [('L4 - infragranular', GRANULAR, DEEP),
                        ('supra - infragranular', UPPER, DEEP),
                        ('supra - granular', UPPER, GRANULAR)]:
    m_vec = (areal_score(mouse_expr, [m_col(g) for g in pos])
             - areal_score(mouse_expr, [m_col(g) for g in neg]))
    h_vec = (areal_score(human_expr, [h_col(g) for g in pos])
             - areal_score(human_expr, [h_col(g) for g in neg]))
    pred = _route_normalized(m_vec, pi)
    keep = cortex & np.isfinite(pred) & np.isfinite(h_vec)
    r_obs = float(pearsonr(pred[keep], h_vec[keep])[0])
    spin = translation_spin_null(m_vec, np.where(cortex, h_vec, np.nan), pi, mouse_coords,
                                 n_trials=1000, seed=0)
    contrasts.append((label, r_obs, float(spin['p_translation_spin'])))

# The upper minus deep contrast, cortex only, against a permuted-pi null of 500 trials.
# 02_layer_marker_refined.py draws every permutation from one generator seeded at 42, and runs
# the three layer-group nulls of 500 trials each before this one. Those draws are taken here so
# the contrast null reads the same part of the stream.
rng = np.random.default_rng(seed=42)
for _ in range(3 * 500):
    rng.permutation(pi.shape[0])

m_contrast = (composite(mouse_expr, [m_col(g) for g in UPPER])
              - composite(mouse_expr, [m_col(g) for g in DEEP]))
h_contrast = (composite(human_expr, [h_col(g) for g in UPPER])
              - composite(human_expr, [h_col(g) for g in DEEP]))
obs_ud = h_contrast[cortex]
r_ud = float(pearsonr((m_contrast @ pi)[cortex], obs_ud)[0])
null_ud = np.array([float(pearsonr((m_contrast @ pi[rng.permutation(pi.shape[0])])[cortex],
                                   obs_ud)[0]) for _ in range(500)])
contrasts.append(('upper - deep', r_ud, float((null_ud >= r_ud).mean())))

mr = np.array([r for _, r, _ in markers]); msig = sum(1 for _, _, q in markers if q < 0.05)
cr = np.array([r for _, r, _ in contrasts]); csig = sum(1 for _, _, q in contrasts if q < 0.05)
print(f"\nlayer markers   n={len(mr)}  mean r {mr.mean():+.3f}  significant {msig}/{len(mr)}")
print(f"layer contrasts n={len(cr)}  mean r {cr.mean():+.3f}  significant {csig}/{len(cr)}\n")
for lab, r, q in contrasts:
    print(f"   {lab:24s} r={r:+.3f}  p={q:.3f}{'   <- granularity is the hierarchy' if 'L4' in lab else ''}")

logged = ([float(m['pearson_r']) for m in mk['markers']]
          + [float(areal[k]['pearson_r']) for k in ('granular_L4_minus_infragranular',
                                                    'supragranular_minus_infragranular',
                                                    'supragranular_minus_granular')]
          + [float(refined['upper_minus_deep_contrast']['pearson_r'])])
gap = np.abs(np.array([r for _, r, _ in markers] + [r for _, r, _ in contrasts]) - logged).max()
print(f"\nlargest gap between a recomputed correlation and its logged value: {gap:.2e}\n")
check('marker mean r', float(mr.mean()))
check('contrast mean r', float(cr.mean()))

## 3. Networks

Routing the mouse resting-state networks of Coletta et al. through π assigns 6 of 10 to their
like-named human network, against a spin-null expectation of 1.0.

In [ ]:
# The count of mouse networks whose routed mass peaks on their like-named human network, and
# the spin null for that count. Both are recomputed from the coupling and the two AnnData
# caches. The aggregation and the diagonal-argmax score are sub-test A of
# experiments/coletta_2020_cross_species_rsn/01_correspondence_validation.py, whose
# labeled_correspondence() is imported and called here rather than restated. The null is the
# mouse-parcel rotation of experiments/spatial_null_check/fair_nulls_coletta_test2c.py, at the
# 500 rotations that script uses. Both logs are still read and compared against.
import os

from scipy.spatial import cKDTree

from otter.data import load_cached
from otter.data.anchors import get_anchor_index
from otter.data.networks import NETWORKS, assign_networks
from otter.eval.nulls import _haar_rotation

col = verified_log('coletta_2020_cross_species_rsn.json')
fair_logged = verified_log('fair_nulls_coletta_test2c.json')['coletta']

CM = load_experiment('coletta_2020_cross_species_rsn/01_correspondence_validation.py')
NC = load_experiment('autism_subtypes/01_network_crossvalidation.py')

M, _ = load_cached('mouse', cache_dir=str(ROOT / 'outputs/anndata'))
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))
mouse_net = assign_networks(M.var, get_anchor_index(M.var))

# assign_human_paper_networks resolves outputs/anndata/_schaefer_order.txt and the Schaefer
# NIfTI against the working directory, so the call runs from ROOT.
cwd = os.getcwd()
os.chdir(ROOT)
try:
    human_net, hnames = NC.assign_human_paper_networks(H.var, separate_aud=True)
finally:
    os.chdir(cwd)
human_net = human_net.copy()
human_net[human_net == hnames.index('Auditory')] = hnames.index('SomatoMotor')

TARGET_PAIRS = [('sensorimotor', 'SomatoMotor'), ('visual', 'Visual'), ('auditory', 'Auditory'),
                ('salience', 'Salience'), ('frontal_dmn', 'DMN'), ('temporal_dmn', 'DMN'),
                ('limbic', 'Limbic'), ('frontoparietal', 'DorsAtten'),
                ('subcortical', 'Subcortical'), ('olfactory', 'Limbic')]

def pair_scores(mouse_labels):
    return CM.labeled_correspondence(pi, mouse_labels, NETWORKS, human_net, hnames,
                                     TARGET_PAIRS)[2]

def n_diagonal(recs):
    return sum(1 for r in recs if r.get('is_argmax_diagonal'))

scored = pair_scores(mouse_net)
sub = {'per_pair_scores': scored,
       'n_diagonal_argmax': n_diagonal(scored),
       'n_pairs_scored': sum('ratio_over_null' in r for r in scored)}

# The mouse parcel centroids are projected to a sphere and rotated, each parcel takes the
# network label of the nearest rotated parcel, and the same coupling is re-aggregated.
mc = M.var[['x', 'y', 'z']].to_numpy(float)
centred = mc - np.nanmean(mc, axis=0)
norms = np.linalg.norm(centred, axis=1, keepdims=True)
norms[norms == 0] = 1.0
sph = centred / norms

rng = np.random.default_rng(0)
null_diag = []
for _ in range(500):
    perm = cKDTree(sph @ _haar_rotation(rng).T).query(sph)[1]
    null_diag.append(n_diagonal(pair_scores(mouse_net[perm])))
null_diag = np.array(null_diag)

fair = {'observed': sub['n_diagonal_argmax'],
        'spin_null_mean': float(null_diag.mean()),
        'spin_p': float((np.sum(null_diag >= sub['n_diagonal_argmax']) + 1) / 501)}

logged_sub = col['sub_test_A_labeled_correspondence']
print(f"\n{sub['n_diagonal_argmax']}/{sub['n_pairs_scored']} mouse networks top-match their homologue")
print(f"spin null mean {fair['spin_null_mean']:.2f}   p = {fair['spin_p']:.3f}")
print(f"logged {logged_sub['n_diagonal_argmax']}/{logged_sub['n_pairs_scored']}, spin null mean "
      f"{fair_logged['spin_null_mean']:.2f}, p = {fair_logged['spin_p']:.3f}\n")
check('networks top-matching homologue', sub['n_diagonal_argmax'])
check('network spin p', fair['spin_p'])

for p in sub['per_pair_scores']:
    mark = 'on homologue' if p['is_argmax_diagonal'] else f"drifts -> {p['argmax_human_net']}"
    print(f"   {p['mouse_net']:14s} {mark}")

## 4. Summary

Translation follows the areal hierarchy. Properties that vary along that axis come across: the
hierarchy maps themselves, cell-class densities that track it, and the FC gradient. Properties
orthogonal to it do not, with laminar contrasts and spatially uniform cell classes all failing
their spin nulls. The dissociation is the result rather than any single correlation.

In [ ]:
print(f"coupling              {PROV['pi_file']}")
print(f"myelin translation    r = {abs(r_t1t2[0]):.2f} (T1w:T2w), {abs(r_cyto[0]):.2f} (cytoarchitecture)")
print(f"routed territory      {n_regions} of 400 Schaefer regions")
print(f"network correspondence {sub['n_diagonal_argmax']}/{sub['n_pairs_scored']}, spin p = {fair['spin_p']:.3f}")
print()
ok = all([check('myelin from mouse T1w:T2w (r)', abs(r_t1t2[0])),
          check('myelin from mouse cytoarch (r)', abs(r_cyto[0])),
          check('routed Schaefer regions', n_regions),
          check('networks top-matching homologue', sub['n_diagonal_argmax']),
          check('network spin p', fair['spin_p'])])
print('\nALL CHECKS PASS' if ok else '\nSOME CHECKS FAILED; text and code have diverged')